# AWS Braket Cost Estimator – IonQ Forte-1

Reads the configuration and circuit-building code directly from `submit_job.py`,
analyses the transpiled gate count, and estimates the task cost.

**Pricing model used here (verify at [AWS Braket Pricing](https://aws.amazon.com/braket/pricing/)):**
- Fixed **per-task fee** – charged once per `backend.run()` call
- **Per-shot fee** – charged for every shot executed on the QPU

> AWS also enforces a practical gate×shot limit on IonQ devices.  
> `submit_job.py` reduces `n_shots` automatically when `n_gates × n_shots > GATE_SHOT_LIMIT`;
> this notebook reproduces that logic so the estimate reflects what would actually be submitted.

In [ ]:
import importlib.util, os, sys
import numpy as np
from qiskit import transpile
from qiskit_aer import AerSimulator

## 1 — Load config and helpers from `submit_job.py`

In [ ]:
SUBMIT_PATH = os.path.join(os.getcwd(), 'submit_job.py')

spec = importlib.util.spec_from_file_location('submit_job', SUBMIT_PATH)
sj   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sj)

# ── Config read from submit_job.py ──────────────────────────────────────────
DEVICE_NAME    = sj.DEVICE_NAME
DEVICE_ARN     = sj.DEVICE_ARN
N_PREC         = sj.N_PREC
N_SHOTS        = sj.N_SHOTS
SEED           = sj.SEED
GATE_SHOT_LIMIT = sj.GATE_SHOT_LIMIT

print('Loaded config from submit_job.py')
print(f'  Device         : {DEVICE_NAME}')
print(f'  N_PREC         : {N_PREC}')
print(f'  N_SHOTS        : {N_SHOTS}')
print(f'  SEED           : {SEED}')
print(f'  GATE_SHOT_LIMIT: {GATE_SHOT_LIMIT:,}')

## 2 — Build circuit and analyse gate counts

The circuit is built with the same functions as `submit_job.py`.  
Because we do not have the real IonQ backend locally, transpilation is done against
a basis gate set that approximates IonQ native gates (`rz`, `ry`, `rxx`).  
The actual gate count on hardware may differ slightly due to device-specific optimisation.

In [ ]:
rng = np.random.default_rng(seed=SEED)
_, angles = sj.haar_random_1qubit_matrix(rng=rng)

qc = sj.build_full_circuit(N_PREC, angles)

# Transpile to an IonQ-like basis gate set as a local approximation
IONQ_BASIS = ['rz', 'ry', 'rxx']   # rxx ≈ Mølmer–Sørensen (2-qubit native)
qc_t = transpile(qc, basis_gates=IONQ_BASIS, optimization_level=1)

ops          = qc_t.count_ops()
n_gates      = qc_t.size()          # total gate count
n_2q_gates   = ops.get('rxx', 0)    # two-qubit gates (most expensive)
depth        = qc_t.depth()
n_qubits     = qc_t.num_qubits

print(f'Circuit stats (transpiled, basis={IONQ_BASIS}):')
print(f'  Qubits          : {n_qubits}')
print(f'  Total gates     : {n_gates}')
print(f'  2-qubit gates   : {n_2q_gates}  (rxx / MS)')
print(f'  Depth           : {depth}')
print(f'  Gate breakdown  : {dict(ops)}')

## 3 — Shot reduction check

`submit_job.py` reduces `n_shots` when `n_gates × n_shots > GATE_SHOT_LIMIT`.
This cell reproduces that logic so the cost estimate uses the **effective** shot count.

In [ ]:
n_shots_effective = N_SHOTS

if n_gates * N_SHOTS > GATE_SHOT_LIMIT:
    n_shots_effective = GATE_SHOT_LIMIT // n_gates
    print(f'WARNING: gates × shots = {n_gates} × {N_SHOTS} = {n_gates * N_SHOTS:,}')
    print(f'         exceeds GATE_SHOT_LIMIT ({GATE_SHOT_LIMIT:,}).')
    print(f'         Shots reduced: {N_SHOTS} → {n_shots_effective}')
else:
    print(f'OK: gates × shots = {n_gates} × {N_SHOTS} = {n_gates * N_SHOTS:,}')
    print(f'    within GATE_SHOT_LIMIT ({GATE_SHOT_LIMIT:,}). No shot reduction.')

print(f'\nEffective shots to be submitted: {n_shots_effective}')

## 4 — Pricing parameters

Edit the values below to match current AWS Braket pricing.  
Verify at: https://aws.amazon.com/braket/pricing/

| Fee | IonQ Harmony | IonQ Aria | IonQ Forte-1 |
|---|---|---|---|
| Per task | $0.30 | $0.30 | $0.30 |
| Per shot | $0.01 | $0.03 | check console |

In [ ]:
# ── Edit these to match current AWS Braket pricing ───────────────────────────
TASK_FEE  = 0.30    # USD, fixed per backend.run() call
SHOT_FEE  = 0.03    # USD per shot  ← verify for IonQ Forte-1
# ─────────────────────────────────────────────────────────────────────────────

N_TASKS   = 1       # number of tasks you plan to submit

## 5 — Cost estimate

In [ ]:
task_cost  = N_TASKS * TASK_FEE
shot_cost  = N_TASKS * n_shots_effective * SHOT_FEE
total_cost = task_cost + shot_cost

print('=' * 48)
print(f'  Cost estimate  –  {DEVICE_NAME}')
print('=' * 48)
print(f'  N_PREC               : {N_PREC}')
print(f'  Requested shots      : {N_SHOTS}')
print(f'  Effective shots      : {n_shots_effective}')
print(f'  Total gates (approx) : {n_gates}')
print(f'  Tasks                : {N_TASKS}')
print()
print(f'  Task fee   ({N_TASKS} × ${TASK_FEE:.2f})    : ${task_cost:.4f}')
print(f'  Shot fee   ({n_shots_effective} × ${SHOT_FEE:.4f})  : ${shot_cost:.4f}')
print('  ' + '-' * 38)
print(f'  Estimated total                : ${total_cost:.4f}')
print('=' * 48)
print()
print('NOTE: gate counts are approximated using a local')
print('      transpilation. Actual hardware gate counts may')
print('      differ. Verify pricing at aws.amazon.com/braket/pricing/')

## 6 — Cost vs N_SHOTS sweep

Shows how total cost scales with shot count, and marks where shot reduction kicks in.

In [ ]:
import matplotlib.pyplot as plt

shot_values     = list(range(50, 1001, 50))
effective_shots = [min(s, GATE_SHOT_LIMIT // n_gates) for s in shot_values]
costs           = [TASK_FEE + e * SHOT_FEE for e in effective_shots]

reduction_threshold = GATE_SHOT_LIMIT // n_gates

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(shot_values, costs, marker='o', markersize=4, label='estimated cost')
ax.axvline(reduction_threshold, color='r', linestyle='--',
           label=f'shot reduction kicks in (>{reduction_threshold})')
ax.axvline(N_SHOTS, color='g', linestyle=':', label=f'current N_SHOTS={N_SHOTS}')
ax.set_xlabel('Requested N_SHOTS')
ax.set_ylabel('Estimated cost (USD)')
ax.set_title(f'Cost vs shots  –  {DEVICE_NAME}  (n_prec={N_PREC}, {n_gates} gates)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Gate-shot limit cap: shots are capped at {reduction_threshold} for this circuit.')